# F008.3 — Multilingual paraphrase + translated similarity hardening (Track B: similarity)

Resumable on a fresh Colab VM (re-run cells 1-12, then the launch cell). Drive `MyDrive/turniting/f008_3/`. F008 / F008.2 are read-only:
R1 continues from a COPY of the F008.2 B_RERANK reranker; D1 fine-tunes BGE-M3. BM25 is always kept (exact copy). Sushi is never trained on
(LONG_DISTANCE_PARAPHRASE_REGRESSION). FINAL = FLORES devtest + Opusparcus test + XNLI test (no Claude rows). Everything EXPERIMENTAL.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Paths
import os, json, shutil, subprocess, hashlib
T = '/content/drive/MyDrive/turniting'
D = f'{T}/f008_3'
for s in ['datasets/raw', 'datasets/pairs', 'datasets/splits', 'datasets/manifests', 'models/biencoder', 'models/reranker', 'models/checkpoints',
          'models/final', 'evaluation', 'reports', 'logs']:
    os.makedirs(f'{D}/{s}', exist_ok=True)
REPO, BRANCH = '/content/Turniting-f0083', 'exp/f008-3-multilingual-paraphrase-hardening'

In [ ]:
# 3. Clone or update the repository (code only)
if not os.path.exists(f'{REPO}/.git'):
    !git clone -q --branch {BRANCH} https://github.com/Anthgg/Turniting.git {REPO}
else:
    !git -C {REPO} fetch -q origin {BRANCH} && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} reset -q --hard origin/{BRANCH}
%cd /content/Turniting-f0083/training
!git log --oneline -1

In [ ]:
# 4. Dependencies (sentence-transformers trainer API; Colab torch kept)
!pip -q install -U "sentence-transformers>=5.0" datasets rank_bm25 langid
import sentence_transformers, transformers; print('sentence-transformers', sentence_transformers.__version__, '| transformers', transformers.__version__)

In [ ]:
# 5-9. GPU: name, VRAM, CUDA, bf16 / fp16 (the training scripts autodetect the same way and probe the batch size)
import torch
assert torch.cuda.is_available(), 'switch the runtime to a GPU'
p = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported()
print('GPU =', p.name); print('VRAM =', f'{p.total_memory/2**30:.1f} GiB'); print('CUDA =', torch.version.cuda)
print('PRECISION =', 'bf16' if BF16 else 'fp16 (+ loss scaling)')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 10. Historical inputs (read-only)
need = {'F008.2 B_RERANK (baseline + R1 init)': f'{T}/f008_2/models/final/f008_2_locked_reranker',
        'F008.2 train (replay)': f'{T}/f008_2/datasets/splits/f008_2_train.parquet',
        'Sushi (regression)': f'{T}/f008_2/datasets/splits/f008_2_final_external_holdout.parquet',
        'PAWS es test (regression)': f'{T}/f008_2/datasets/splits/f008_2_regression_paws_test.parquet'}
for k, v in need.items():
    print('OK ' if os.path.exists(v) else 'MISSING', k)
assert all(os.path.exists(v) for v in need.values())

In [ ]:
# 11-12. Manifests, hashes and leakage (re-run any time)

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(8 << 20), b''): h.update(c)
    return h.hexdigest()

M, S = f'{D}/datasets/manifests', f'{D}/datasets/splits'
hm = f'{M}/f008_3_final_external_holdout.json'
if os.path.exists(hm):
    h = json.load(open(hm))
    print('FINAL LOCKED:', h['LOCKED'], '| sha ok:', sha(f'{S}/f008_3_final_external_holdout.parquet') == h['file_sha256'])
    print('leakage:', json.dumps(h['leakage']))
else:
    print('splits not frozen yet')

In [ ]:
# Launch f0083_pipeline.sh detached (survives cell interrupts; never use `!cmd &` and never interrupt the cell that launched a job)
pid = '/content/f0083_pipeline.pid'
if os.path.exists(pid) and os.path.exists(f"/proc/{open(pid).read().strip()}"):
    print('already running')
else:
    out = open('/content/f0083_pipeline.out', 'a')
    pr = subprocess.Popen(['bash', 'scripts/f0083_pipeline.sh'], stdout=out, stderr=subprocess.STDOUT, start_new_session=True)
    open(pid, 'w').write(str(pr.pid)); print('started pid', pr.pid)

In [ ]:
# Monitor
!tail -n 30 /content/f0083_pipeline.out 2>/dev/null; echo; ls {D}/logs
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

In [ ]:
# Report bundle for Git (download and hand it to Claude)
!python -W ignore scripts/f0083_report.py
!tar czf {D}/f008_3_reports_bundle.tgz -C /content/Turniting-f0083/training reports/f008_3 F008_3_FINAL_REPORT.md F008_3_FINAL_REPORT.json 2>/dev/null; ls -la {D}/f008_3_reports_bundle.tgz
from google.colab import files
files.download(f'{D}/f008_3_reports_bundle.tgz')